In [1]:
# Частина 2 — Individual Household Electric Power Consumption

import pandas as pd
import numpy as np
import timeit

In [2]:
# Завантаження датасету
df2 = pd.read_csv('household_power_consumption.txt', 
                  sep=';', 
                  na_values=['?'],
                  low_memory=False)

print(df2.shape)
df2.head()

(2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [3]:
# Data Cleaning
def clean_power_df(df):
    # Видаляємо рядки з пропусками
    df = df.dropna()
    
    # Об'єднуємо Date і Time в один стовпець datetime
    df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
    
    # Видаляємо непотрібні стовпці
    df = df.drop(columns=['Date', 'Time'])
    
    # Переставляємо DateTime першим
    cols = ['DateTime'] + [c for c in df.columns if c != 'DateTime']
    df = df[cols]
    
    return df

df2 = clean_power_df(df2)
print(df2.shape)
df2.head()

(2049280, 8)


,DateTime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [4]:
# Вибірка 1 — Global_active_power > 5 кВт
def select_high_power(df):
    result = df[df['Global_active_power'] > 5]
    return result

t = timeit.timeit(lambda: select_high_power(df2), number=10)
print(f"Час виконання: {t:.4f} сек")
print(f"Кількість записів: {len(select_high_power(df2))}")
select_high_power(df2).head()

Час виконання: 0.0456 сек
Кількість записів: 17547


,DateTime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,2006-12-16 17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,2006-12-16 17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


In [5]:
# Вибірка 2 — сила струму 19-20А, пральна+холодильник > бойлер+кондиціонер
def select_current_and_metering(df):
    # Відбираємо записи з силою струму 19-20А
    filtered = df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)]
    result = filtered[filtered['Sub_metering_2'] > filtered['Sub_metering_3']]
    return result

t = timeit.timeit(lambda: select_current_and_metering(df2), number=10)
print(f"Час виконання: {t:.4f} сек")
print(f"Кількість записів: {len(select_current_and_metering(df2))}")
select_current_and_metering(df2).head()

Час виконання: 0.0706 сек
Кількість записів: 2509


,DateTime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,2006-12-16 18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,2006-12-17 01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,2006-12-17 01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,2006-12-17 01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,2006-12-17 01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


In [6]:
# Вибірка 3 — 500000 випадкових записів, середні значення груп споживання
def select_random_500k(df):
    sample = df.sample(n=500000, replace=False)
    means = {
        'Sub_metering_1 (середнє)': sample['Sub_metering_1'].mean(),
        'Sub_metering_2 (середнє)': sample['Sub_metering_2'].mean(),
        'Sub_metering_3 (середнє)': sample['Sub_metering_3'].mean(),
    }
    return sample, means

t = timeit.timeit(lambda: select_random_500k(df2), number=3)
print(f"Час виконання: {t:.4f} сек")
sample, means = select_random_500k(df2)
for k, v in means.items():
    print(f"{k}: {v:.4f}")
sample.head()

Час виконання: 0.5086 сек
Sub_metering_1 (середнє): 1.1216
Sub_metering_2 (середнє): 1.2936
Sub_metering_3 (середнє): 6.4516


,DateTime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
703561,2008-04-18 07:25:00,0.870,0.066,240.14,3.6,0.0,0.0,1.0
924383,2008-09-18 15:47:00,0.362,0.084,242.00,1.6,0.0,0.0,1.0
242047,2007-06-02 19:31:00,3.694,0.290,230.63,16.0,37.0,0.0,16.0
339116,2007-08-09 05:20:00,0.216,0.120,234.25,1.0,0.0,0.0,0.0
409387,2007-09-27 00:31:00,0.384,0.206,239.83,1.8,0.0,1.0,0.0


In [7]:
# Вибірка 4 — після 18:00, понад 6 кВт, Sub_metering_2 найбільша, кожен 3й і 4й
def select_evening_heavy(df):
    # Після 18:00 і більше 6 кВт
    filtered = df[
        (df['DateTime'].dt.hour >= 18) & 
        (df['Global_active_power'] > 6)
    ]
    # Sub_metering_2 є найбільшою групою
    filtered = filtered[
        (filtered['Sub_metering_2'] > filtered['Sub_metering_1']) & 
        (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])
    ]
    # Ділимо на дві половини
    half = len(filtered) // 2
    first_half = filtered.iloc[:half]
    second_half = filtered.iloc[half:]
    # Кожен 3й з першої, кожен 4й з другої
    result = pd.concat([
        first_half.iloc[::3],
        second_half.iloc[::4]
    ])
    return result
t = timeit.timeit(lambda: select_evening_heavy(df2), number=10)
print(f"Час виконання: {t:.4f} сек")
result4 = select_evening_heavy(df2)
print(f"Кількість записів: {len(result4)}")
result4.head()

Час виконання: 0.4593 сек
Кількість записів: 310


,DateTime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,2006-12-16 18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,2006-12-16 18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,2006-12-28 20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,2006-12-28 21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,2006-12-28 21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


In [8]:
# Нормалізація та стандартизація
def normalize_standardize(df):
    cols = ['Global_active_power', 'Global_reactive_power', 
            'Voltage', 'Global_intensity',
            'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    # Нормалізація (min-max)
    df_normalized = df[cols].copy()
    for col in cols:
        min_val = df[col].min()
        max_val = df[col].max()
        df_normalized[col] = (df[col] - min_val) / (max_val - min_val)
    # Стандартизація (z-score)
    df_standardized = df[cols].copy()
    for col in cols:
        mean = df[col].mean()
        std = df[col].std()
        df_standardized[col] = (df[col] - mean) / std
    return df_normalized, df_standardized
df_norm, df_std = normalize_standardize(df2)
print("Нормалізовані дані:")
print(df_norm.head(3))
print("\nСтандартизовані дані:")
print(df_std.head(3))

Нормалізовані дані:
   Global_active_power  Global_reactive_power   Voltage  Global_intensity  \
0             0.374796               0.300719  0.376090          0.377593   
1             0.478363               0.313669  0.336995          0.473029   
2             0.479631               0.358273  0.326010          0.473029   

   Sub_metering_1  Sub_metering_2  Sub_metering_3  
0             0.0          0.0125        0.548387  
1             0.0          0.0125        0.516129  
2             0.0          0.0250        0.548387  

Стандартизовані дані:
   Global_active_power  Global_reactive_power   Voltage  Global_intensity  \
0             2.955076               2.610720 -1.851816          3.098788   
1             4.037084               2.770405 -2.225274          4.133799   
2             4.050325               3.320431 -2.330213          4.133799   

   Sub_metering_1  Sub_metering_2  Sub_metering_3  
0       -0.182337       -0.051274        1.249420  
1       -0.182337       -0.

In [9]:
# Коефіцієнти кореляції Пірсона та Спірмена
from scipy import stats

def correlation_coefficients(df):
    col1 = 'Global_active_power'
    col2 = 'Global_intensity'
    
    pearson, p_pearson = stats.pearsonr(df[col1], df[col2])
    spearman, p_spearman = stats.spearmanr(df[col1], df[col2])
    
    print(f"Пірсон:  {pearson:.4f} (p-value: {p_pearson:.4f})")
    print(f"Спірмен: {spearman:.4f} (p-value: {p_spearman:.4f})")

correlation_coefficients(df2)

Пірсон:  0.9989 (p-value: 0.0000)
Спірмен: 0.9954 (p-value: 0.0000)


In [10]:
# One Hot Encoding категоріального атрибута
def one_hot_encoding(df):
    # Створюємо категоріальний атрибут по годинах доби
    df = df.copy()
    df['time_of_day'] = pd.cut(
        df['DateTime'].dt.hour,
        bins=[0, 6, 12, 18, 24],
        labels=['ніч', 'ранок', 'день', 'вечір'],
        right=False
    )
    
    # One Hot Encoding
    encoded = pd.get_dummies(df['time_of_day'], prefix='time')
    result = pd.concat([df[['DateTime', 'Global_active_power']], encoded], axis=1)
    return result

result_ohe = one_hot_encoding(df2)
print(result_ohe.head(10))

             DateTime  Global_active_power  time_ніч  time_ранок  time_день  \
0 2006-12-16 17:24:00                4.216     False       False       True   
1 2006-12-16 17:25:00                5.360     False       False       True   
2 2006-12-16 17:26:00                5.374     False       False       True   
3 2006-12-16 17:27:00                5.388     False       False       True   
4 2006-12-16 17:28:00                3.666     False       False       True   
5 2006-12-16 17:29:00                3.520     False       False       True   
6 2006-12-16 17:30:00                3.702     False       False       True   
7 2006-12-16 17:31:00                3.700     False       False       True   
8 2006-12-16 17:32:00                3.668     False       False       True   
9 2006-12-16 17:33:00                3.662     False       False       True   

   time_вечір  
0       False  
1       False  
2       False  
3       False  
4       False  
5       False  
6       False  
7 